In [ ]:
import subprocess, os

# FIX 8 (kept for context): Use env vars with fallback so notebook runs outside Kaggle too
TMP     = os.environ.get('TMP_DIR', '/kaggle/tmp')
WORKING = os.environ.get('WORKING_DIR', '/kaggle/working')
os.makedirs(TMP, exist_ok=True)
os.makedirs(WORKING, exist_ok=True)

def download_if_missing(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print(f"Already exists: {dest}")
        return
    subprocess.run(['curl', '-L', url, '-o', dest])

download_if_missing(
    'https://zenodo.org/records/6390798/files/embryo_dataset_annotations.tar.gz',
    f'{TMP}/annotations.tar.gz'
)

download_if_missing(
    'https://zenodo.org/records/6390798/files/embryo_dataset.tar.gz',
    f'{TMP}/images.tar.gz'
)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 53267  100 53267    0     0  68387      0 --:--:-- --:--:-- --:--:-- 68378
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
 17 11.3G   17 2008M    0     0  16.1M      0  0:11:59  0:02:04  0:09:55 16.8M

In [ ]:
import tarfile

def extract_if_needed(tar_path, out):
    if os.path.exists(out) and os.listdir(out):
        print("Already extracted:", out)
        return
    os.makedirs(out, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall(out)

extract_if_needed(f'{TMP}/annotations.tar.gz', f'{TMP}/annotations')
extract_if_needed(f'{TMP}/images.tar.gz', f'{TMP}/images')

def resolve_root(p):
    x = os.listdir(p)
    if len(x)==1:
        return os.path.join(p,x[0])
    return p

IMAGE_ROOT = resolve_root(f'{TMP}/images')
ANNOTATION_ROOT = resolve_root(f'{TMP}/annotations')

print(IMAGE_ROOT, ANNOTATION_ROOT)

In [ ]:
import random, glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 5
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-3

In [ ]:
PHASE_ORDER = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6','t7','t8',
    't9','tM','tSB','tB','tEB'
]

CLASS_TO_IDX = {k:i for i,k in enumerate(PHASE_ORDER)}
NUM_CLASSES = len(CLASS_TO_IDX)

all_videos = sorted(os.listdir(IMAGE_ROOT))
random.seed(42)

remove = set(random.sample(all_videos, int(0.3*len(all_videos))))
videos = [v for v in all_videos if v not in remove]

print("Kept videos:", len(videos))

In [ ]:
def build_sequences():
    video_to_sequences = {}

    for csv_file in os.listdir(ANNOTATION_ROOT):
        if not csv_file.endswith('.csv'):
            continue

        vid = csv_file.replace('_phases.csv','')
        if vid not in videos:
            continue

        folder = os.path.join(IMAGE_ROOT, vid)
        csv_path = os.path.join(ANNOTATION_ROOT, csv_file)

        if not os.path.exists(folder):
            continue

        # map RUN -> filename
        run_map = {}
        for f in os.listdir(folder):
            if 'RUN' in f:
                try:
                    idx = int(f.split('RUN')[-1].split('.')[0])
                    run_map[idx] = f
                except:
                    pass

        df = pd.read_csv(csv_path, header=None)

        # FIX 2: Build per-phase frame lists first, then create sequences
        # within each phase only — prevents sequences from crossing phase boundaries.
        phase_frames = {}  # phase_label -> list of (run_idx, path)

        for _, row in df.iterrows():
            phase, s, e = row[0], int(row[1]), int(row[2])
            if phase not in CLASS_TO_IDX:
                continue

            label = CLASS_TO_IDX[phase]

            if label not in phase_frames:
                phase_frames[label] = []

            for i in range(s, e + 1):
                if i in run_map:
                    phase_frames[label].append((i, os.path.join(folder, run_map[i])))

        seqs = []
        for label, frames in phase_frames.items():
            # Sort by run index within each phase
            frames.sort(key=lambda x: x[0])

            # Slide window strictly within this phase
            for i in range(len(frames) - SEQ_LEN + 1):
                chunk = frames[i:i + SEQ_LEN]
                paths = [x[1] for x in chunk]
                seqs.append((paths, label))

        if len(seqs):
            video_to_sequences[vid] = seqs

    return video_to_sequences

video_to_sequences = build_sequences()

total_seq = sum(len(v) for v in video_to_sequences.values())
print("Total sequences:", total_seq)

In [ ]:
from sklearn.model_selection import train_test_split

vids = list(video_to_sequences.keys())

train_vids, temp_vids = train_test_split(vids, test_size=0.3, random_state=42)
val_vids, test_vids  = train_test_split(temp_vids, test_size=0.5, random_state=42)

train_seq, val_seq, test_seq = [], [], []

for v in train_vids:
    train_seq.extend(video_to_sequences[v])

for v in val_vids:
    val_seq.extend(video_to_sequences[v])

for v in test_vids:
    test_seq.extend(video_to_sequences[v])

print("Train:", len(train_seq))
print("Val:", len(val_seq))
print("Test:", len(test_seq))

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class DatasetSeq(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        paths, label = self.data[idx]

        imgs = []
        for p in paths:
            # FIX 4: Catch specific exceptions and log them instead of silently swallowing all errors
            try:
                img = Image.open(p).convert('RGB')
                img = transform(img)
            except (FileNotFoundError, OSError, Exception) as e:
                print(f"[WARNING] Failed to load image {p}: {e}")
                img = torch.zeros(3, 224, 224)
            imgs.append(img)

        return torch.stack(imgs), torch.tensor(label)

# FIX 7: Use num_workers=0 for portability across Kaggle, Windows, and restricted environments.
# Change to num_workers=2 only if you are on Linux with no multiprocessing restrictions.
train_loader = DataLoader(DatasetSeq(train_seq), batch_size=16, shuffle=True,  num_workers=0)
val_loader   = DataLoader(DatasetSeq(val_seq),   batch_size=16, shuffle=False, num_workers=0)
test_loader  = DataLoader(DatasetSeq(test_seq),  batch_size=16, shuffle=False, num_workers=0)

In [ ]:
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights='IMAGENET1K_V1')
        self.cnn = nn.Sequential(*list(base.children())[:-1])
        self.lstm = nn.LSTM(512,128,batch_first=True)
        self.fc = nn.Linear(128, NUM_CLASSES)

    def forward(self,x):
        B,T,C,H,W = x.shape
        x = x.view(B*T,C,H,W)
        f = self.cnn(x).squeeze(-1).squeeze(-1)
        f = f.view(B,T,512)
        o,_ = self.lstm(f)
        return self.fc(o[:,-1,:])

model = CNN_LSTM().to(DEVICE)

In [ ]:
import numpy as np

# ----------------------------
# Compute class weights
# ----------------------------
class_counts = np.zeros(NUM_CLASSES)

for _, label in train_seq:
    class_counts[label] += 1

# FIX 3: Use standard inverse-frequency weighting without re-normalizing.
# This preserves the correct relative magnitude of weights for CrossEntropyLoss.
total_samples = class_counts.sum()
weights = total_samples / (NUM_CLASSES * (class_counts + 1e-6))

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", weights)

# ----------------------------
# Loss with weights
# ----------------------------
criterion = nn.CrossEntropyLoss(weight=weights)

# ----------------------------
# Optimizer + Scheduler
# ----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.3
)

best_loss = 1e9
patience = 4
counter = 0

log_file = f"{WORKING}/result.txt"

def log(s):
    print(s)
    with open(log_file,"a") as f:
        f.write(s+"\n")

for epoch in range(EPOCHS):
    model.train()
    tl = 0

    for x,y in tqdm(train_loader):
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()
        tl += loss.item()

    model.eval()
    vl, correct, total = 0,0,0

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out,y)
            vl += loss.item()

            p = out.argmax(1)
            correct += (p==y).sum().item()
            total += y.size(0)
    vl = vl / len(val_loader)
    tl = tl / len(train_loader)
    acc = correct/total
    log(f"Epoch {epoch+1} TL={tl:.3f} VL={vl:.3f} ACC={acc:.4f}")

    scheduler.step(vl)

    if vl < best_loss:
        best_loss = vl
        torch.save(model.state_dict(), f"{WORKING}/best_model.pth")
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        break

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# FIX 5 + FIX 6: Added weights_only=True (safe loading in PyTorch >= 2.0)
# and map_location=DEVICE so the model loads correctly on CPU or GPU.
model.load_state_dict(
    torch.load(f"{WORKING}/best_model.pth", map_location=DEVICE, weights_only=True)
)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        preds = torch.argmax(out, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ----------------------------
# Confusion Matrix
# ----------------------------
cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))

# ----------------------------
# Per-class accuracy
# ----------------------------
per_class_acc = {}

for i in range(NUM_CLASSES):
    total = cm[i].sum()
    correct = cm[i][i]

    acc = correct / total if total > 0 else 0.0
    per_class_acc[i] = acc

# ----------------------------
# Overall accuracy
# ----------------------------
overall_acc = np.sum(np.diag(cm)) / np.sum(cm)

print("FINAL TEST ACC:", overall_acc)

# ----------------------------
# Save everything
# ----------------------------
with open(f"{WORKING}/result.txt", "a") as f:
    f.write("\n===== FINAL TEST RESULTS =====\n")
    f.write(f"Overall Accuracy: {overall_acc:.4f}\n\n")

    f.write("Confusion Matrix:\n")
    for row in cm:
        f.write(" ".join(map(str, row)) + "\n")

    f.write("\nPer-class Accuracy:\n")
    for i in range(NUM_CLASSES):
        class_name = PHASE_ORDER[i]
        f.write(f"{class_name}: {per_class_acc[i]:.4f}\n")

print("Saved confusion matrix + per-class accuracy to result.txt ✅")